### Data collection notebook

In [18]:
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
from urllib.request import urlretrieve
from zipfile import ZipFile

# ─── Config ───────────────────────────────────────────────────────
BASE_DIR    = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
BASE_URL    = "https://s3.amazonaws.com/tripdata/"

ZIP_DIR     = os.path.join(BASE_DIR, "zips")
EXTRACT_DIR = os.path.join(BASE_DIR, "csv")

os.makedirs(ZIP_DIR,     exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

print(f"Data will be saved in:\n  Zips   → {ZIP_DIR}\n  CSVs   → {EXTRACT_DIR}\n")

# ─── Determine latest month & 14-month window ─────────────────────
latest_yyyymm = "202601"

latest_dt = datetime.strptime(latest_yyyymm + "01", "%Y%m%d")
start_dt  = latest_dt - relativedelta(months=13)

target_months = []
current = start_dt
while current <= latest_dt:
    target_months.append(current.strftime("%Y%m"))
    current += relativedelta(months=1)

print("Target JC months:", target_months)
print("-" * 50)

# ─── Download + Extract ───────────────────────────────────────────
for yyyymm in target_months:
    candidates = [
        f"JC-{yyyymm}-citibike-tripdata.csv.zip",
        f"JC-{yyyymm}-citibike-tripdata.zip"
    ]
    
    downloaded = False
    for filename in candidates:
        zip_url  = BASE_URL + filename
        zip_path = os.path.join(ZIP_DIR, filename)
        
        if os.path.exists(zip_path):
            print(f"Already have {filename} → skipping download")
            downloaded = True
            break
        
        try:
            print(f"Trying {filename} ...")
            urlretrieve(zip_url, zip_path)
            print(f"Downloaded {filename}")
            downloaded = True
            break
        except Exception as e:
            err_str = str(e).lower()
            if "404" in err_str or "not found" in err_str or "does not exist" in err_str:
                continue  # try next filename variant
            else:
                print(f"  → Error on {filename}: {e}")
                break
    
    if not downloaded:
        print(f"No file found for {yyyymm} (tried both naming variants)")
        continue
    
    # Extract
    try:
        with ZipFile(zip_path, 'r') as z:
            csvs = [f for f in z.namelist() if f.lower().endswith('.csv')]
            if not csvs:
                print(f"No CSV found inside {os.path.basename(zip_path)}")
                continue
            
            # Extract only CSV files (cleaner than extractall)
            for csv_name in csvs:
                target_csv_path = os.path.join(EXTRACT_DIR, os.path.basename(csv_name))
                if os.path.exists(target_csv_path):
                    print(f"  → Already extracted: {os.path.basename(csv_name)}")
                    continue
                z.extract(csv_name, EXTRACT_DIR)
                print(f"  → Extracted: {os.path.basename(csv_name)}")
            
            print(f"Finished extracting from {os.path.basename(zip_path)}\n")
            
    except Exception as e:
        print(f"Extraction failed for {os.path.basename(zip_path)}: {e}\n")

# ─── Summary ──────────────────────────────────────────────────────
csv_files = [f for f in os.listdir(EXTRACT_DIR) if f.lower().endswith('.csv')]
csv_files.sort()

print("=" * 70)
print(f"Done! Found {len(csv_files)} JC CSV files in:")
print(f"  {EXTRACT_DIR}")
print(f"Latest 5 files:")
for f in csv_files[-5:]:
    print(f"  • {f}")
print("=" * 70)

Data will be saved in:
  Zips   → c:\Users\aghab\Desktop\coding\citibike\data\zips
  CSVs   → c:\Users\aghab\Desktop\coding\citibike\data\csv

Target JC months: ['202412', '202501', '202502', '202503', '202504', '202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512', '202601']
--------------------------------------------------
Trying JC-202412-citibike-tripdata.csv.zip ...
Downloaded JC-202412-citibike-tripdata.csv.zip
  → Extracted: JC-202412-citibike-tripdata.csv
  → Extracted: ._JC-202412-citibike-tripdata.csv
Finished extracting from JC-202412-citibike-tripdata.csv.zip

Trying JC-202501-citibike-tripdata.csv.zip ...
Downloaded JC-202501-citibike-tripdata.csv.zip
  → Extracted: JC-202501-citibike-tripdata.csv
  → Extracted: ._JC-202501-citibike-tripdata.csv
Finished extracting from JC-202501-citibike-tripdata.csv.zip

Trying JC-202502-citibike-tripdata.csv.zip ...
Downloaded JC-202502-citibike-tripdata.csv.zip
  → Extracted: JC-202502-citibike-tripdata.csv
  